# HW2 Part I — REINFORCE, step by step

This notebook takes apart `src/course_tasks/hw2/reinforce.py`, the REINFORCE
implementation you'll turn into PPO in Part II, alongside **Sutton & Barto,
Chapter 13** and **Williams (1992)**. Do the Step 2 reading in `hw2/README.md`
first.

* It runs the **real HW2 environment on the login node's CPU** — zero GPU-hours.
* It is **not graded and not submitted**. It exists so you understand the code
  before you change it.
* Cells marked **TODO** ask for a line or two. The cell after each one checks
  your answer against `reinforce.py`. Questions marked **(writeup)** are answered
  in your HW2 writeup.

Run the cells in order. If you restart the kernel, start again from the top.

## 0. Setup

Select the repo's `.venv` as the kernel (see `docs/01_workflow.md`). The import
of `course_tasks` is what registers the `Course-HW2-*` tasks.

In [ ]:
import copy
import os

os.environ.setdefault("MJLAB_WARP_QUIET", "1")

import matplotlib.pyplot as plt
import torch
from torch.distributions import Normal

import course_tasks  # noqa: F401  registers the Course-* tasks
from course_tasks.hw2.modules import Critic, GaussianActor
from course_tasks.hw2.reinforce import Reinforce
from course_tasks.hw2.storage import RolloutStorage
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg

torch.manual_seed(0)
print("CUDA available:", torch.cuda.is_available(), "(False is expected on the login node)")


def todo():
  raise NotImplementedError("Fill in this TODO, then re-run the cell.")

## 1. The sources in one table

| Williams (1992) | Sutton & Barto | HW2 code |
|---|---|---|
| weights $w_{ij}$ | policy parameter $\theta$ | `actor` parameters |
| unit output $y$ | action $A_t$ | `storage.actions` |
| output distribution $g$ | policy $\pi(a \mid s, \theta)$ | `GaussianActor.distribution` |
| reinforcement $r$ | return $G_t$ | `storage.returns` |
| reinforcement baseline $b_{ij}$ | baseline $b(S_t)$, e.g. $\hat{v}(S_t, \mathbf{w})$ | `storage.values` |
| characteristic eligibility $\partial \ln g / \partial w_{ij}$ | eligibility vector $\nabla \ln \pi(A_t \mid S_t, \theta)$ | autograd through `evaluate_actions` |
| learning rate factor $\alpha_{ij}$ | step size $\alpha$ | `learning_rate` |

Williams' §4 update is $\Delta w_{ij} = \alpha_{ij}(r - b_{ij})\,e_{ij}$. S&B's
REINFORCE update (eq. 13.8) is the same idea with the return $G_t$ in place of
$r$. In a network trained by gradient descent we write it as a loss,
$-\,(G_t - b_t)\log\pi_\theta(A_t \mid S_t)$, and let autograd compute the
eligibility for every weight at once.

## 2. The policy: a Gaussian unit (Williams §6, S&B §13.7)

`GaussianActor` is Williams' §6 Gaussian unit, and S&B's eq. 13.19 policy, with a
neural network in front of it: an MLP computes the mean $\mu(s)$, and the
standard deviation is a learned parameter. Williams gives the eligibilities of
the mean and standard deviation in closed form — eq. 12 for $\mu$, and the
unnumbered equation just after it for $\sigma$.

**TODO:** write both eligibilities by hand. The check compares them to autograd.

In [ ]:
mu = torch.tensor([0.3], requires_grad=True)
sigma = torch.tensor([0.8], requires_grad=True)
a = torch.tensor([1.5])  # an action the unit "took"

log_g = Normal(mu, sigma).log_prob(a).sum()
log_g.backward()  # autograd's answer lands in mu.grad and sigma.grad

with torch.no_grad():
  # TODO: Williams eq. 12 — the characteristic eligibility of mu.
  elig_mu = todo()
  # TODO: the eligibility of sigma, just after eq. 12.
  elig_sigma = todo()

In [ ]:
assert torch.allclose(elig_mu, mu.grad, atol=1e-6), (elig_mu, mu.grad)
assert torch.allclose(elig_sigma, sigma.grad, atol=1e-6), (elig_sigma, sigma.grad)
print("Both eligibilities match autograd.")

Williams' footnote 2 of §6 points out that nothing stops a gradient step from
making $\sigma$ negative, and suggests adapting $\ln\sigma$ instead. S&B's
eq. 13.20 parameterizes $\sigma$ through an exponential for the same reason, and
their Exercise 13.4 asks for the eligibility vector of exactly this kind of
policy. That is why `GaussianActor` stores `log_std`. **(writeup, Q3)** What is
$\partial \ln \pi / \partial (\ln \sigma)$?

Now the real actor. `act` samples; `evaluate_actions` re-scores given actions.
Log-probabilities are summed over the action dimension (here there's only one).

In [ ]:
actor_demo = GaussianActor(obs_dim=5, action_dim=1)
obs_demo = torch.randn(4, 5)
actions_demo, log_probs_demo = actor_demo.act(obs_demo)
print("actions:", actions_demo.shape, " log_probs:", log_probs_demo.shape)
print("std:", actor_demo.std.item(), "(learned; starts at init_std=1.0)")

## 3. The environment

The warm-up task: a single cartpole that starts near upright and must stay up.
This builds it with 32 parallel environments on the CPU. The first build compiles
simulation kernels and takes a little while.

In [ ]:
N_ENVS = 32
env_cfg = load_env_cfg("Course-HW2-Cartpole-Balance-Reinforce")
env_cfg.scene.num_envs = N_ENVS
env_cfg.seed = 0
env = RslRlVecEnvWrapper(ManagerBasedRlEnv(cfg=env_cfg, device="cpu"))

obs_td = env.get_observations()
OBS_DIM = obs_td["actor"].shape[-1]
T = int(env.max_episode_length)
print("observation groups:", list(obs_td.keys()))
print("actor observation:", tuple(obs_td["actor"].shape), "= cart pos, cos/sin pole, cart vel, pole vel")
print("actions:", env.num_actions)
print("control step:", env.unwrapped.step_dt, "s   steps per episode:", T)

Two facts that shape everything below:

* **An episode is exactly 100 control steps** (5 s at 0.05 s), and the task is
  finite-horizon: step 100 is a true terminal.
* **Rewards are multiplied by the 0.05 s step**, so the best possible episode
  scores about **5.0**.

## 4. Collect one rollout

The same loop `runner.py` runs. Collection happens under
`torch.inference_mode()`: fast, and nothing produced inside carries gradients.

In [ ]:
actor = GaussianActor(OBS_DIM, env.num_actions)
critic = Critic(OBS_DIM)
storage = RolloutStorage(T, N_ENVS, OBS_DIM, env.num_actions, device="cpu")
alg = Reinforce(actor, critic, storage, gamma=0.99, baseline="value")


def collect(alg, env):
  """One rollout of storage.num_transitions_per_env steps. Returns the last observation."""
  obs = env.get_observations()["actor"]
  alg.storage.clear()
  with torch.inference_mode():
    for _ in range(alg.storage.num_transitions_per_env):
      actions = alg.act(obs)
      obs_td, rewards, dones, _ = env.step(actions)
      obs = obs_td["actor"]
      alg.process_env_step(rewards, dones)
  return obs


last_obs = collect(alg, env)
print("rewards:", tuple(storage.rewards.shape), " [T, N]")
print("steps where any env was done:", torch.nonzero(storage.dones.sum(1)).flatten().tolist())
print("mean episode return:", storage.rewards.sum(0).mean().item(), "(max about 5.0)")

Every environment finished at the last index, 99, and nowhere else: one column of
the buffer is one complete trajectory. That is what makes a Monte-Carlo return
exact here — and it is the restriction PPO will let you drop.

## 5. Returns

$G_t = r_t + \gamma\,(1 - \text{done}_t)\,G_{t+1}$, computed backwards. The
$(1 - \text{done}_t)$ factor stops one episode's return leaking into the
previous episode.

**TODO:** implement it for a `[T, N]` batch. Loop over time only.

In [ ]:
def returns_to_go(rewards, dones, gamma):
  """rewards, dones: [T, N] -> returns [T, N]."""
  # TODO: the backwards recursion above.
  return todo()

In [ ]:
toy_r = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
toy_d = torch.tensor([[0.0], [1.0], [0.0], [1.0]])  # an episode ends at t=1
assert torch.allclose(returns_to_go(toy_r, toy_d, 0.5).squeeze(), torch.tensor([2.0, 2.0, 5.0, 4.0]))

alg.compute_returns(last_obs)
assert torch.allclose(returns_to_go(storage.rewards, storage.dones, 0.99), storage.returns)
print("Matches Reinforce.compute_returns.")

Try it: how does $\gamma$ reshape the return along one episode?

In [ ]:
for gamma in (0.9, 0.99, 1.0):
  plt.plot(returns_to_go(storage.rewards, storage.dones, gamma)[:, 0], label=f"gamma={gamma}")
plt.xlabel("t"); plt.ylabel("G_t (env 0)"); plt.legend(); plt.show()

## 6. Total return, return-to-go, and a baseline — measured

Williams' episodic REINFORCE (§5, eq. 11) weights *every* step's eligibility by
the *whole* episode's reinforcement. S&B's REINFORCE (§13.3) and `reinforce.py`
weight step $t$ by the return from $t$ onward. Williams' §4 and S&B's §13.4 both
allow subtracting a baseline, as long as it does not depend on the action.

All three give an unbiased gradient. They differ in **variance**, which you can
measure: collect several independent batches from the *same* policy, compute the
gradient estimate from each, and look at how much those estimates disagree.

Undiscounted ($\gamma = 1$) throughout — the same setting S&B's chapter text
uses — so only the weighting differs.

In [ ]:
K = 8
batches = []
for _ in range(K + 1):
  collect(alg, env)
  batches.append((storage.observations.clone(), storage.actions.clone(),
                  storage.rewards.clone(), storage.dones.clone()))
# The first batch only supplies the baseline, so the baseline is independent of
# the actions in the batches it is used on.
ref_batch, batches = batches[0], batches[1:]
baseline_t = returns_to_go(ref_batch[2], ref_batch[3], 1.0).mean(dim=1)  # [T]
print(f"collected {K} batches of {N_ENVS} episodes each")

**TODO:** the three weightings. Each returns a `[T, N]` tensor of weights for the
log-probabilities.

In [ ]:
def weights_total_return(rewards, dones):
  # TODO: Williams eq. 11 — every step gets the whole episode's return.
  return todo()


def weights_return_to_go(rewards, dones):
  # TODO: step t gets the return from t onward (gamma = 1).
  return todo()


def weights_return_to_go_minus_baseline(rewards, dones):
  # TODO: the return from t onward, minus baseline_t (shape [T]).
  return todo()

In [ ]:
def gradient_estimate(obs, actions, weights):
  actor.zero_grad()
  log_probs, _ = actor.evaluate_actions(obs.reshape(-1, OBS_DIM), actions.reshape(-1, env.num_actions))
  (-(log_probs * weights.reshape(-1)).mean()).backward()
  return torch.cat([p.grad.flatten() for p in actor.parameters()]).clone()


results = {}
for name, fn in [("total return (eq. 11)", weights_total_return),
                 ("return-to-go", weights_return_to_go),
                 ("return-to-go - baseline", weights_return_to_go_minus_baseline)]:
  grads = torch.stack([gradient_estimate(o, a, fn(r, d)) for o, a, r, d in batches])
  results[name] = grads.var(dim=0).sum().item()

for name, v in results.items():
  print(f"{name:26s} total gradient variance {v:.3e}")

**(writeup, Q2)** Why is the return-to-go estimate still unbiased, even though
it throws away rewards from before step $t$? Use the numbers above.

## 7. The update, and the `inference_mode` trap

The stored log-probabilities came out of `inference_mode`. Here are the two ways
reusing them in a loss fails.

In [ ]:
obs0, acts0 = storage.observations[0], storage.actions[0]
with torch.inference_mode():
  _, lp_from_collection = actor.act(obs0)

try:
  (-lp_from_collection.mean()).backward()
except RuntimeError as e:
  print("1) backward through a collected log-prob:\n  ", e)

try:
  lp_now, _ = actor.evaluate_actions(obs0, acts0)
  (-(lp_now * lp_from_collection).mean()).backward()
except RuntimeError as e:
  print("2) mixing it into a differentiable computation:\n  ", e)

**(writeup, Q4)** Explain both errors. The fix `update()` uses: re-score the
stored actions with `evaluate_actions` under the current policy. That copy
carries gradients.

**TODO:** the policy-gradient loss, given the re-scored log-probs and the
advantages. The check runs `Reinforce.update` on the same batch.

In [ ]:
last_obs = collect(alg, env)
alg.compute_returns(last_obs)
alg.compute_advantages()
obs, actions, old_log_probs, returns, advantages = storage.flatten()
log_probs, entropy = actor.evaluate_actions(obs, actions)

In [ ]:
# TODO: the REINFORCE policy loss (a scalar). Optimizers minimize; we want to
# increase the probability of actions with positive advantage.
policy_loss = todo()

In [ ]:
reference = Reinforce(copy.deepcopy(actor), copy.deepcopy(critic), storage, baseline="value").update()
assert abs(policy_loss.item() - reference["policy"]) < 1e-5, (policy_loss.item(), reference["policy"])
print("Matches Reinforce.update. (Advantages come out of compute_advantages with no graph attached.)")

## 8. Put it together: train briefly

The whole algorithm, as `runner.py` runs it: collect, compute returns and
advantages, update. A couple of minutes on the CPU with 32 environments — enough
to see the return climb, not to solve the task.

In [ ]:
N_ITERS = 25
history = []
for it in range(N_ITERS):
  last_obs = collect(alg, env)
  history.append(storage.rewards.sum(0).mean().item())
  alg.compute_returns(last_obs)
  alg.compute_advantages()
  losses = alg.update()
  if it % 5 == 0 or it == N_ITERS - 1:
    print(f"iter {it:2d}  return {history[-1]:.2f}  std {actor.std.item():.3f}  policy loss {losses['policy']:+.4f}")

plt.plot(history); plt.xlabel("iteration"); plt.ylabel("mean episode return (max ~5.0)"); plt.show()

## 9. Toward PPO

Open the PPO paper at §2.1. Eq. 2,

$$L^{PG}(\theta) = \hat{\mathbb{E}}_t\big[\log \pi_\theta(a_t \mid s_t)\,\hat{A}_t\big],$$

is the loss you just wrote (negated, because optimizers minimize). The paragraph
after it is the reason PPO exists: taking many optimization steps on this loss
with the same batch "often leads to destructively large policy updates".
REINFORCE takes one step and throws the data away.

S&B's §13.5 (actor–critic) shows the other ingredient PPO uses: replacing the
full return with a bootstrapped estimate from a learned value function. PPO's
advantage estimator (its eq. 11–12) is built from exactly that.

Part II starts there. `ppo.py` is `reinforce.py` with a different class name —
go to Step 5 of `hw2/README.md`.

In [ ]:
env.close()  # free the environment; also shut down the kernel when you're done